In [28]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

# Data splitting
from sklearn.model_selection import train_test_split

# Text processing
from sklearn.feature_extraction.text import TfidfVectorizer

# ML pipeline
from sklearn.pipeline import Pipeline

# Classification models
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

# XGBoost
from xgboost import XGBClassifier

# Label encoding
from sklearn.preprocessing import LabelEncoder

# Evaluation metrics
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    ConfusionMatrixDisplay
)

print("OpsMind AI - Ticket Classification")
print("Libraries Imported Successfully!")

OpsMind AI - Ticket Classification
Libraries Imported Successfully!


In [29]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [30]:
df = pd.read_csv("../data/processed/feature_engineered_incidents.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (48549, 19)


,title,body,ticket_type,category,sub_category1,sub_category2,business_service,urgency,impact,title_missing,title_length,body_length,title_word_count,body_word_count,has_password,has_network,has_email,has_access,body_missing
0,NaN,hi since recruiter lead permission approve req...,1,4,2,21,71,3,4,0,NaN,90,NaN,12,0,0,0,0,0
1,connection with icon,icon dear please setup icon per icon engineers...,1,6,22,7,26,3,4,0,20.0,90,3.0,15,0,0,0,0,0
2,work experience user,work experience user hi work experience studen...,1,5,13,7,32,3,4,0,20.0,103,3.0,16,0,0,0,0,0
3,requesting for meeting,requesting meeting hi please help follow equip...,1,5,13,7,32,3,4,0,22.0,70,3.0,11,0,0,0,0,0
4,reset passwords for external accounts,re expire days hi ask help update passwords co...,1,4,2,76,4,3,4,0,37.0,910,5.0,140,1,0,0,1,0


In [31]:
print("Category Distribution:")
print(df["category"].value_counts().sort_values(ascending=False))

print("\nNumber of Categories:", df["category"].nunique())

Category Distribution:
category
4     34061
5      9634
6      2628
7       921
11      612
8       239
9       191
3       137
1        72
12       45
0         4
2         3
10        2
Name: count, dtype: int64

Number of Categories: 13


In [32]:
# Combine title and body for text-based classification
df["text"] = (
    df["title"].fillna("") + " " +
    df["body"].fillna("")
).str.strip()

# Remove records with empty text
df = df[df["text"] != ""].copy()

# Input feature and target
X = df["text"]
y = df["category"]

print("Total Samples:", len(df))
print("Input Features:", X.shape)
print("Target Classes:", y.nunique())
print("\nTarget Distribution:")
print(y.value_counts().sort_index())


Total Samples: 48549
Input Features: (48549,)
Target Classes: 13

Target Distribution:
category
0         4
1        72
2         3
3       137
4     34061
5      9634
6      2628
7       921
8       239
9       191
10        2
11      612
12       45
Name: count, dtype: int64


In [33]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

Training Samples: 38839
Testing Samples: 9710


In [34]:
# LOGISTIC REGRESSION MODEL
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        stop_words="english",
        min_df=2
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)

print("Model Training Completed Successfully!")

Model Training Completed Successfully!


In [35]:
#LOGISTIC REGRESSION MODEL EVALUATION

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    zero_division=0
))

Accuracy: 0.8181256436663233

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.06      0.29      0.10        14
           2       0.00      0.00      0.00         1
           3       0.08      0.26      0.12        27
           4       0.95      0.84      0.89      6812
           5       0.80      0.77      0.78      1927
           6       0.38      0.76      0.51       526
           7       0.54      0.89      0.67       184
           8       0.71      0.77      0.74        48
           9       0.35      0.76      0.48        38
          11       0.52      0.77      0.62       123
          12       0.50      0.56      0.53         9

    accuracy                           0.82      9710
   macro avg       0.41      0.56      0.45      9710
weighted avg       0.87      0.82      0.84      9710



In [37]:
# COMPARE WITH OTHER MODELS MULTINOMIAL NAIVE BAYES, COMPLEMENT NAIVE BAYES, LINEAR SVM
models = {
    "Multinomial Naive Bayes": MultinomialNB(),
    "Complement Naive Bayes": ComplementNB(),
    "Linear SVM": LinearSVC(class_weight="balanced")
}

results = []

for name, classifier in models.items():

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=20000,
            ngram_range=(1, 2),
            stop_words="english",
            min_df=2
        )),
        ("classifier", classifier)
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Macro F1": f1_score(
            y_test, predictions,
            average="macro",
            zero_division=0
        ),
        "Weighted F1": f1_score(
            y_test, predictions,
            average="weighted",
            zero_division=0
        )
    })

results.append({
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Macro F1": f1_score(
        y_test, y_pred,
        average="macro",
        zero_division=0
    ),
    "Weighted F1": f1_score(
        y_test, y_pred,
        average="weighted",
        zero_division=0
    )
})

comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.sort_values(
    by="Macro F1", ascending=False
).reset_index(drop=True)

comparison_df

,Model,Accuracy,Macro F1,Weighted F1
0,Linear SVM,0.864058,0.500556,0.866669
1,Logistic Regression,0.818126,0.453769,0.836248
2,Complement Naive Bayes,0.857158,0.416574,0.854944
3,Multinomial Naive Bayes,0.838311,0.281335,0.810062


In [ ]:
# Additional Models: Random Forest and SGD Classifier
extra_models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    ),

    "SGD Classifier": SGDClassifier(
        loss="modified_huber",
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    )
}

extra_results = []

for name, classifier in extra_models.items():

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=20000,
            ngram_range=(1, 2),
            stop_words="english",
            min_df=2
        )),
        ("classifier", classifier)
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    extra_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Macro F1": f1_score(
            y_test, predictions,
            average="macro",
            zero_division=0
        ),
        "Weighted F1": f1_score(
            y_test, predictions,
            average="weighted",
            zero_division=0
        )
    })

extra_results_df = pd.DataFrame(extra_results)
extra_results_df

,Model,Accuracy,Macro F1,Weighted F1
0,Random Forest,0.824099,0.390807,0.793995
1,SGD Classifier,0.861071,0.494242,0.865148


In [ ]:
# XGBOOST TRAINING MODEL

# Convert text into TF-IDF features
tfidf_xgb = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words="english",
    min_df=2
)

X_train_xgb = tfidf_xgb.fit_transform(X_train)
X_test_xgb = tfidf_xgb.transform(X_test)

# Encode target labels
encoder = LabelEncoder()
y_train_xgb = encoder.fit_transform(y_train)
y_test_xgb = encoder.transform(y_test)

print("TF-IDF Shape:", X_train_xgb.shape)
print("XGBoost Data Prepared!")

TF-IDF Shape: (38839, 10000)
XGBoost Data Prepared!


In [ ]:
# XGBoost Classifier

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    objective="multi:softmax",
    num_class=len(encoder.classes_),
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train_xgb, y_train_xgb)

xgb_pred_encoded = xgb_model.predict(X_test_xgb)
xgb_pred = encoder.inverse_transform(xgb_pred_encoded)

print("XGBoost Accuracy:", accuracy_score(y_test, xgb_pred))
print("XGBoost Macro F1:", f1_score(
    y_test, xgb_pred, average="macro", zero_division=0
))
print("XGBoost Weighted F1:", f1_score(
    y_test, xgb_pred, average="weighted", zero_division=0
))

XGBoost Accuracy: 0.8592173017507724
XGBoost Macro F1: 0.47015854609434876
XGBoost Weighted F1: 0.8457989939411183


In [ ]:
xgb_result = pd.DataFrame([{
    "Model": "XGBoost",
    "Accuracy": accuracy_score(y_test, xgb_pred),
    "Macro F1": f1_score(
        y_test, xgb_pred,
        average="macro",
        zero_division=0
    ),
    "Weighted F1": f1_score(
        y_test, xgb_pred,
        average="weighted",
        zero_division=0
    )
}])

final_comparison = pd.concat(
    [all_results, xgb_result],
    ignore_index=True
)

final_comparison = final_comparison.sort_values(
    "Macro F1", ascending=False
).reset_index(drop=True)

final_comparison

,Model,Accuracy,Macro F1,Weighted F1
0,Linear SVM,0.864058,0.500556,0.866669
1,SGD Classifier,0.861071,0.494242,0.865148
2,XGBoost,0.859217,0.470159,0.845799
3,Logistic Regression,0.818126,0.453769,0.836248
4,Complement Naive Bayes,0.857158,0.416574,0.854944
5,Random Forest,0.824099,0.390807,0.793995
6,Multinomial Naive Bayes,0.838311,0.281335,0.810062


In [38]:
# Create a machine learning pipeline
# TF-IDF converts ticket text into numerical features
# Linear SVM performs ticket category classification

best_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=20000,       # Maximum number of text features
        ngram_range=(1, 2),       # Use single words and two-word combinations
        stop_words="english",     # Remove common English words
        min_df=2                  # Ignore words appearing in fewer than 2 documents
    )),

    ("classifier", LinearSVC(
        class_weight="balanced",  # Handle imbalanced categories
        random_state=42           # Keep results reproducible
    ))
])

# Train the complete pipeline using training data
best_model.fit(X_train, y_train)

print("Best Linear SVM Model Trained Successfully!")

Best Linear SVM Model Trained Successfully!


In [39]:
# Save the complete pipeline
# It includes both TF-IDF vectorizer and trained Linear SVM model

joblib.dump(
    best_model,
    "../data/processed/best_ticket_classifier.pkl"
)

print("Best Model Saved Successfully!")

Best Model Saved Successfully!


In [40]:
# Load the saved ticket classification pipeline
loaded_model = joblib.load(
    "../data/processed/best_ticket_classifier.pkl"
)

# Prepare sample IT support tickets
sample_tickets = [
    "unable to access company email account",
    "network connection is not working",
    "request for new laptop purchase"
]

# Predict categories for sample tickets
predictions = loaded_model.predict(sample_tickets)

# Display prediction results
for ticket, category in zip(sample_tickets, predictions):
    print("Ticket:", ticket)
    print("Predicted Category:", category)
    print("-" * 50)

Ticket: unable to access company email account
Predicted Category: 4
--------------------------------------------------
Ticket: network connection is not working
Predicted Category: 6
--------------------------------------------------
Ticket: request for new laptop purchase
Predicted Category: 5
--------------------------------------------------


# Conclusion

In this notebook, we developed a machine learning-based
IT ticket category classification system.

### Models Evaluated
1. Linear SVM
2. SGD Classifier
3. Logistic Regression
4. Complement Naive Bayes
5. Random Forest
6. Multinomial Naive Bayes
7. XGBoost

### Best Model
Linear Support Vector Machine (Linear SVM)

### Best Model Performance
- Accuracy: 86.41%
- Macro F1-score: 0.501
- Weighted F1-score: 0.867

### Key Findings
- Linear SVM achieved the best overall benchmark scores.
- Class imbalance affects minority category performance.
- TF-IDF was used for text feature extraction.
- The trained model was saved as a reusable pipeline.
- Sample ticket predictions were successfully tested.

### Future Improvements
- Improve minority category classification.
- Perform hyperparameter tuning.
- Add model explainability.
- Integrate the model with FastAPI.